# Stage 1 Revision: Episode-Level Arrhythmia Features


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path("..").resolve()

ANNOTATION_DIR = BASE_DIR / "external_data" / "vitaldb-arrhythmia-database-1.0.0" / "Annotation_Files"

DATA_INTERIM = BASE_DIR / "data" / "interim"
CLINICAL_CSV = DATA_INTERIM / "imputed_477_cases.csv"
FEATURES_CSV = DATA_INTERIM / "arrhythmia_features_482.csv"  # stage-1 output, to be updated

EPISODES_DRAFT_CSV = DATA_INTERIM / "arrhythmia_episodes_draft.csv"
FEATURES_UPDATED_DRAFT_CSV = DATA_INTERIM / "arrhythmia_features_updated_draft.csv"

print(f"Annotation folder exists: {ANNOTATION_DIR.exists()}")
print(f"Clinical CSV exists: {CLINICAL_CSV.exists()}")
print(f"Existing features CSV exists: {FEATURES_CSV.exists()}")

In [ ]:
MIN_CLEAN_ROWS = 10        # same quality gate as the original stage 1 extraction
RR_MIN_SEC = 0.2
RR_MAX_SEC = 3.0
MIN_VALID_RR = 3
EPISODE_GAP_SEC = 30.0      # a new episode starts when the gap exceeds this many seconds
MIN_EPISODE_BEATS = 3       # episodes with fewer non-normal beats than this are dropped entirely

## Episode Extraction Function

Reuses the same quality filter and the same `beat_type`/`rhythm_label` `NaN`-guarding logic discovered while building the original `arrhythmia_features_482.csv` (clean rows can still have a missing `beat_type`, and a non-normal `beat_type` row can still have a missing `rhythm_label` — both need explicit `.notna()` checks before any `!= 'N'` comparison, since `NaN != 'N'` is `True` in pandas).

`episode_dominant_rhythm` excludes `rhythm_label == 'N'` the same way the patient-level `dominant_rhythm` column did (most frequent *non-normal* label; falls back to `'N'` only if every beat in the episode happens to carry rhythm_label `'N'` despite having a non-normal `beat_type`, which does occur in the real data).

In [ ]:
def extract_episodes(case_id):
    """
    Returns (episodes, n_excluded_episodes, n_excluded_beats) for one patient.
    episodes is a list of dicts, one per VALID episode (>= MIN_EPISODE_BEATS beats).
    """
    ann_path = ANNOTATION_DIR / f"Annotation_file_{case_id}.csv"
    if not ann_path.exists():
        print(f"WARNING: annotation file missing for caseid {case_id} - 0 episodes recorded")
        return [], 0, 0

    ann = pd.read_csv(ann_path)

    # Same Step 1 quality filter as the original extraction
    clean = ann[ann["bad_signal_quality"] == False].copy()
    if len(clean) < MIN_CLEAN_ROWS:
        print(f"WARNING: caseid {case_id} has only {len(clean)} clean rows - 0 episodes recorded")
        return [], 0, 0

    clean = clean.sort_values("time_second")

    # Non-normal beats only - this is the population episodes are built from
    non_normal = clean[clean["beat_type"].notna() & (clean["beat_type"] != "N")].copy()
    if len(non_normal) == 0:
        return [], 0, 0

    non_normal = non_normal.sort_values("time_second")

    # Cluster consecutive non-normal beats into raw episode groups: a new group starts
    # whenever the gap to the previous non-normal beat exceeds EPISODE_GAP_SEC
    times = non_normal["time_second"].values
    gaps = np.diff(times)
    raw_episode_id = np.zeros(len(times), dtype=int)
    raw_episode_id[1:] = (gaps > EPISODE_GAP_SEC).cumsum()
    non_normal["raw_episode_id"] = raw_episode_id

    episodes = []
    n_excluded_episodes = 0
    n_excluded_beats = 0
    valid_episode_number = 0

    for raw_id, group in non_normal.groupby("raw_episode_id"):
        if len(group) < MIN_EPISODE_BEATS:
            # Isolated ectopic beat(s) - not a real episode, drop entirely
            n_excluded_episodes += 1
            n_excluded_beats += len(group)
            continue

        valid_episode_number += 1
        ep_start = group["time_second"].min()
        ep_end = group["time_second"].max()

        # episode_dominant_rhythm: most frequent NON-NORMAL rhythm_label
        rl = group["rhythm_label"].dropna()
        rl_non_normal = rl[rl != "N"]
        if len(rl_non_normal) > 0:
            ep_dominant_rhythm = rl_non_normal.value_counts().index[0]
        elif len(rl) > 0:
            ep_dominant_rhythm = "N"
        else:
            ep_dominant_rhythm = np.nan

        # episode_beat_type: most frequent of S or V only (excludes U/P)
        bt_sv = group["beat_type"][group["beat_type"].isin(["S", "V"])]
        ep_beat_type = bt_sv.value_counts().index[0] if len(bt_sv) > 0 else np.nan

        # episode_rr_cv: RR variability among this episode's non-normal beats only
        sorted_times = np.sort(group["time_second"].values)
        rr = np.diff(sorted_times)
        valid_rr = rr[(rr >= RR_MIN_SEC) & (rr <= RR_MAX_SEC)]
        if len(valid_rr) >= MIN_VALID_RR:
            ep_rr_cv = np.std(valid_rr) / np.mean(valid_rr)
        else:
            ep_rr_cv = np.nan

        episodes.append({
            "caseid": case_id,
            "episode_number": valid_episode_number,
            "episode_start_sec": ep_start,
            "episode_end_sec": ep_end,
            "episode_duration_sec": ep_end - ep_start,
            "episode_beat_count": len(group),
            "episode_dominant_rhythm": ep_dominant_rhythm,
            "episode_beat_type": ep_beat_type,
            "episode_rr_cv": ep_rr_cv,
        })

    return episodes, n_excluded_episodes, n_excluded_beats


# Quick sanity check on case 337
eps, n_exc, n_exc_beats = extract_episodes(337)
print(f"Case 337: {len(eps)} valid episodes, {n_exc} excluded ({n_exc_beats} isolated beats)")

In [ ]:
clinical = pd.read_csv(CLINICAL_CSV)
case_ids = clinical["caseid"].tolist()
print(f"Processing {len(case_ids)} caseids "
      f"(gap threshold={EPISODE_GAP_SEC}s, min beats per episode={MIN_EPISODE_BEATS})")

all_episodes = []
total_excluded_episodes = 0
total_excluded_beats = 0

for cid in case_ids:
    eps, n_exc, n_exc_beats = extract_episodes(cid)
    all_episodes.extend(eps)
    total_excluded_episodes += n_exc
    total_excluded_beats += n_exc_beats

episodes_df = pd.DataFrame(all_episodes)
print(f"\nTotal VALID episodes remaining: {len(episodes_df)}")
print(f"Episodes EXCLUDED for having fewer than {MIN_EPISODE_BEATS} beats: {total_excluded_episodes} "
      f"(representing {total_excluded_beats} total isolated non-normal beats filtered out)")
print(f"\nepisodes_df shape: {episodes_df.shape}")
print(episodes_df.head(10).to_string())

In [ ]:
# Episode count distribution per patient (0, 1, 2-5, 5+)
episode_counts_per_patient = episodes_df.groupby("caseid").size()
# Patients with zero valid episodes don't appear in the groupby above - add them back as 0
episode_counts_per_patient = episode_counts_per_patient.reindex(case_ids, fill_value=0)

bins = pd.cut(episode_counts_per_patient, bins=[-0.1, 0, 1, 5, np.inf], labels=["0", "1", "2-5", "5+"])
print("Episode count distribution across all 477 patients:")
print(bins.value_counts().sort_index())

## Build Per-Patient Summary Columns

In [ ]:
summary = episodes_df.groupby("caseid").agg(
    total_episode_count=("episode_number", "count"),
    total_arrhythmia_burden_sec=("episode_duration_sec", "sum"),
    longest_episode_duration_sec=("episode_duration_sec", "max"),
    first_episode_start_sec=("episode_start_sec", "min"),
).reset_index()

# Left-merge onto the FULL patient list so patients with zero valid episodes are kept,
# rather than silently disappearing because groupby only sees patients who have rows
all_ids_df = pd.DataFrame({"caseid": case_ids})
summary = all_ids_df.merge(summary, on="caseid", how="left")

# Sum of an empty set is legitimately 0, so these two are safe to fill with 0.
# "longest" and "first" of an empty set are undefined, so those stay NaN.
summary["total_episode_count"] = summary["total_episode_count"].fillna(0).astype(int)
summary["total_arrhythmia_burden_sec"] = summary["total_arrhythmia_burden_sec"].fillna(0.0)

print(f"summary shape: {summary.shape}")
print(f"Patients with 0 valid episodes: {(summary['total_episode_count']==0).sum()}")
print(summary.head(10).to_string())

## Merge Into the Existing arrhythmia_features_482.csv

`event_time_sec` is dropped and replaced by `first_episode_start_sec` per the new instructions. Everything else in the existing file (`rhythm_onset_normal`, `arrhythmia_duration_sec`, beat composition, whole-file `rr_cv`/`rr_mean`) is preserved unchanged — those describe the whole annotated segment and remain useful alongside the new per-episode breakdown.

In [ ]:
features = pd.read_csv(FEATURES_CSV)
print(f"Existing features shape: {features.shape}")

features_updated = features.drop(columns=["event_time_sec"]).merge(summary, on="caseid", how="left")
print(f"Updated features shape: {features_updated.shape}")
print(f"Updated columns: {list(features_updated.columns)}")

In [ ]:
assert episodes_df["caseid"].notna().all()
assert features_updated["caseid"].is_unique, "caseid must stay unique"

episodes_df.to_csv(EPISODES_DRAFT_CSV, index=False)
features_updated.to_csv(FEATURES_UPDATED_DRAFT_CSV, index=False)
print(f"Saved DRAFT episodes table to {EPISODES_DRAFT_CSV.resolve()}")
print(f"Saved DRAFT updated features table to {FEATURES_UPDATED_DRAFT_CSV.resolve()}")
print("Run verify_arrhythmia_episodes.ipynb next to check these drafts before they become final.")